# 01 — Data Pull (Phase 1 gate)

Goal: gather historical Mantle on-chain events for Lendle + Init Capital + Agni + Merchant Moe, plus L1 bridge arrivals, into local parquet cache.

Kill-criterion blocker: if smart-money universe < 500 active Mantle wallets, pivot strategy.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from data_loader import ADDRESSES, get_block_number, get_logs, MANTLE_MAINNET_RPC
from universe import count_universe, fetch_bridge_arrivals
from loguru import logger

## 1. Sanity-check Mantle RPC + addresses

In [ ]:
latest = get_block_number(MANTLE_MAINNET_RPC)
print(f'Mantle Mainnet latest block: {latest:,}')
print(f'Lendle LendingPool: {ADDRESSES.lendle_lending_pool}')
print(f'Init Capital POS_MANAGER: {ADDRESSES.init_pos_manager}')
print(f'Agni Factory: {ADDRESSES.agni_factory}')
print(f'Merchant Moe LB Factory: {ADDRESSES.moe_lb_factory}')
print(f'mETH token: {ADDRESSES.meth_token}')
print(f'USDY token: {ADDRESSES.usdy_token}')

## 2. Phase 0 Gate — count Mantle smart-money universe

Cross-reference Ethereum L1 bridge events to find wallets that arrived on Mantle, then filter by recent Mantle activity.

⚠️ This step may take 10-30 minutes due to RPC pagination. Cache results.

In [ ]:
# DRY-RUN with small window first to validate RPC pagination
wallets = fetch_bridge_arrivals(block_window_days=30)
print(f'Distinct arrivals last 30 days: {len(wallets)}')
Path('../data').mkdir(parents=True, exist_ok=True)
Path('../data/bridge_arrivals_30d.json').write_text(json.dumps(wallets))

In [ ]:
# Full count — 180 day window + active filter
# counts = count_universe(block_window_days=180, activity_window_days=30)
# print(f'Total bridge arrivals (180d): {counts.total_bridge_arrivals}')
# print(f'Active on Mantle (30d): {counts.active_last_30d_on_mantle}')
# print(f'Passes 500-wallet gate: {counts.passes_gate}')

## 3. Resolve event topic0 hashes via keccak

We need canonical topic0 for each event signature so eth_getLogs can filter.

In [ ]:
from eth_utils import keccak

EVENT_SIGS = {
    'Supply': 'Supply(address,address,address,uint256,uint16)',
    'Borrow': 'Borrow(address,address,address,uint256,uint8,uint256,uint16)',
    'Repay': 'Repay(address,address,address,uint256,bool)',
    'LiquidationCall': 'LiquidationCall(address,address,address,uint256,uint256,address,bool)',
    'PoolCreated': 'PoolCreated(address,address,uint24,int24,address)',
    'Swap_AgniV3': 'Swap(address,address,int256,int256,uint160,uint128,int24)',
    'LBPairCreated': 'LBPairCreated(address,address,uint256,uint256)',
}

for name, sig in EVENT_SIGS.items():
    h = '0x' + keccak(text=sig).hex()
    print(f'{name:20s} {h}')


## 4. Pull Lendle events into parquet

Begin with a recent 7-day window to validate the indexer logic, then expand to 90d for backtest.

In [ ]:
# from_block = latest - (7 * 86_400 // 2)  # ~7 days on Mantle (2s block time)
# supply_topic = '0x' + keccak(text=EVENT_SIGS['Supply']).hex()
# logs = get_logs(
#     rpc_url=MANTLE_MAINNET_RPC,
#     contract_address=ADDRESSES.lendle_lending_pool,
#     topic0=supply_topic,
#     from_block=from_block,
#     to_block='latest',
# )
# print(f'Lendle Supply events (7d): {len(logs)}')